In [33]:
import sys
if "tf_env" not in sys.executable:
    print("/n环境配置错误!!!/n")
    print(sys.executable)
else:
    print("环境配置正常")

环境配置正常


In [34]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt 
from matplotlib import rcParams # 字体配置,支持中文
rcParams['font.family'] = 'SimHei'
rcParams['axes.unicode_minus'] = False  # 解决负号不显示的问题

import tensorflow as tf

In [35]:
# ===== 线性回归模型的构建 =====

In [36]:
# 1. 生成数据集:与3-1一样
def synthetic_data(w, b, num_examples):  
    """生成y=Xw+b+噪声"""
    X = tf.zeros((num_examples, w.shape[0]))
    X += tf.random.normal(shape=X.shape)
    y = tf.matmul(X, tf.reshape(w, (-1, 1))) + b
    y += tf.random.normal(shape=y.shape, stddev=0.01)
    y = tf.reshape(y, (-1, 1))
    return X, y

true_w = tf.constant([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

In [37]:
# 2. 读取数据集:使用API
# is_train表示是否希望数据迭代器对象在每个迭代周期内打乱数据
def load_array(data_arrays, batch_size, is_train=True):
    """构造一个TensorFlow数据迭代器"""
    dataset = tf.data.Dataset.from_tensor_slices(data_arrays)
    if is_train:
        dataset = dataset.shuffle(buffer_size=1000)
    dataset = dataset.batch(batch_size)
    return dataset

batch_size = 10
data_iter = load_array((features, labels), batch_size)

In [38]:
# 3. 定义模型: keras是TensorFlow的高级API
net = tf.keras.Sequential()
net.add(tf.keras.layers.Dense(1))

In [39]:
# 4. 初始化模型参数:
# TensorFlow中的initializers模块提供了多种模型参数初始化方法。 在Keras中最简单的指定初始化方法是在创建层时指定kernel_initializer
# 初始化与输入多少个参数当前完全没联系: 我们正在为网络初始化参数，而Keras还不知道输入将有多少维,但是不担心,其会自动检测的
initializer = tf.initializers.RandomNormal(stddev=0.01) # 均值为0、标准差为0.01
net = tf.keras.Sequential()
net.add(tf.keras.layers.Dense(1, kernel_initializer=initializer))

In [40]:
# 5. 定义损失函数
loss = tf.keras.losses.MeanSquaredError()

In [41]:
# 6. 定义优化算法: 小批量随机梯度下降算法: API: SGD只需要设置learning_rate值
trainer = tf.keras.optimizers.SGD(learning_rate=0.03)

In [ ]:
# 7. 训练
'''
在每个迭代周期里,我们将完整遍历一次数据集(train_data), 不停地从中获取一个小批量的输入和相应的标签。 对于每一个小批量，我们会进行以下步骤:
    通过调用net(X)生成预测并计算损失l(前向传播)
    通过进行反向传播来计算梯度。
    通过调用优化器来更新模型参数。
'''
num_epochs = 3  # 训练3轮
for epoch in range(num_epochs):
    for X, y in data_iter:
        # 向前传播,开始求梯度
        with tf.GradientTape() as tape:
            l = loss(net(X, training=True), y)  # net()意思是把自变量X送入神经网络(含有想要求的w/b常量),返回神经网络的预测值y_hat
        grads = tape.gradient(l, net.trainable_variables)   # 求导(梯度)net.trainable_variables是模型里所有可训练参数
        # 使用trainer优化算法进行w/b参数更新(传入梯度(all)和可训练参数(w/b),也就是做了一步:参数 = 参数 - 学习率 × 梯度)
        trainer.apply_gradients(zip(grads, net.trainable_variables))    
    # 得到loss损失(y_hat = net(自变量) - y(labels)) ^ 2
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

# 对比参数
w = net.get_weights()[0]
print('w的估计误差:', true_w - tf.reshape(w, true_w.shape))
b = net.get_weights()[1]
print('b的估计误差:', true_b - b)

epoch 1, loss 0.000104
epoch 2, loss 0.000102
epoch 3, loss 0.000102
epoch 4, loss 0.000103
epoch 5, loss 0.000104
epoch 6, loss 0.000102
epoch 7, loss 0.000102
epoch 8, loss 0.000102
epoch 9, loss 0.000102
epoch 10, loss 0.000102
w的估计误差: tf.Tensor([2.0146370e-05 1.4305115e-04], shape=(2,), dtype=float32)
b的估计误差: [0.00031662]
